In [1]:
#! pip install llama-index llama-index-llms-openai llama-index-vector-stores-chroma
# pip install -U llama-index llama-index-embeddings-google-genai google-genai

In [2]:
from dotenv import load_dotenv

import os

load_dotenv()


True

In [3]:
bool(os.getenv("Google_api_key"))

True

## Document load


In [4]:
import os
from pathlib import Path


data_dire=Path('data')
assert data_dire.exists(),'File doesnt exit '

Api_key=os.getenv("Google_api_key")

In [5]:
from llama_index.core import SimpleDirectoryReader

Documents=SimpleDirectoryReader(data_dire).load_data()

In [6]:
len(Documents)


3

## Chuncking strategy

In [7]:
from llama_index.core.node_parser import TokenTextSplitter
chuncks=TokenTextSplitter(chunk_size=500,chunk_overlap=100)


## Embedding


In [8]:
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

# Initialize Gemini embedding
emb_model = GoogleGenAIEmbedding(
    model="gemini-embedding-2-preview", 
    api_key=Api_key  
)


## Vector store

In [9]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
client=chromadb.EphemeralClient()
collection_name='Entreprise_company'
collection=client.get_or_create_collection(collection_name)
vector_store=ChromaVectorStore(chroma_collection=collection)
print("Chroma collection is ready :",collection)

Chroma collection is ready : Collection(name=Entreprise_company)


## Ingestion Pipeline

In [10]:
from llama_index.core.ingestion import IngestionPipeline
pipeline=IngestionPipeline(transformations=[chuncks,emb_model],vector_store=vector_store)

## Pipeline Execution

In [11]:
chuncks=pipeline.run(documents=Documents)
print('number of chunsk :',len(chuncks))


number of chunsk : 17


# Rag Pipeline

In [12]:
# Create Index 

from llama_index.core import VectorStoreIndex

vector_index= VectorStoreIndex.from_vector_store(embed_model=emb_model,vector_store=vector_store)


In [13]:
from llama_index.llms.gemini import Gemini
llm = Gemini(
    model="gemini-2.5-flash", 
    api_key=Api_key
)

c:\Users\Sachi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Sachi\AppData\Local\Programs\Python\Python311\Lib\site-packages\llama_index\llms\gemini\base.py:21: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
C:\Users\Sachi\AppData\Local\Temp\ipykernel_15572\1862143663.py:2: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm

In [14]:
query_engine= vector_index.as_query_engine(llm=llm,similarity_top_k=2)

In [15]:
query='who is CEo'

response=query_engine.query(query)

print(response.response)
print(response.source_nodes)

Bar is the CEO.
[NodeWithScore(node=TextNode(id_='64026e2b-5b59-4f00-a33f-1fe6f72916c8', embedding=None, metadata={'file_path': 'k:\\Api_and_gen_Ai\\RAG_folder\\data\\About.md', 'file_name': 'About.md', 'file_size': 10152, 'creation_date': '2025-11-03', 'last_modified_date': '2026-02-20'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='465287fd-a2d1-431a-a9bc-f808f57c4472', node_type='4', metadata={'file_path': 'k:\\Api_and_gen_Ai\\RAG_folder\\data\\About.md', 'file_name': 'About.md', 'file_size': 10152, 'creation_date': '2025-11-03', 'last_modified_date': '2026-02-20'}, hash='b7a784a222da93021642c53dbe9dbe0333d10d72495f34868d2072a33e630fdf'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='bf89c0b8

# Evaluation

In [16]:
#! pip install deepeval


## Compliteness

In [17]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.models import GeminiModel
eval_model = GeminiModel(
    model="gemini-2.5-flash",
    api_key=Api_key
)

In [18]:
compliteness=AnswerRelevancyMetric(threshold=0.5,model=eval_model,include_reason=True)
test_case=LLMTestCase(input=query,actual_output=response.response)
print('Quer:',query,"response:",response.response)
compliteness.measure(test_case)
print(compliteness.reason)
print(compliteness.score)


c:\Users\Sachi\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Quer: who is CEo response: Bar is the CEO.


The score is 1.00 because the output is perfectly relevant to the input. Great job!
1.0
